In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

In [5]:
index.search(
    "How does the agentic loop keep calling the model until it stops?",
    num_results=1,
)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [6]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [7]:
answer_output_text, answer_usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print("### Output Text ###")
print(answer_output_text)
print("### Usage ###")
print(answer_usage)


### Output Text ###
The agentic loop keeps calling the model inside a `while True` loop.

Each iteration:
1. sends the full message history to the model,
2. checks whether the model returned any `function_call`s,
3. runs those tools and appends the results to memory,
4. repeats.

It stops when a turn comes back with no function calls:

```python
if has_function_calls == False:
    break
```

So the loop continues until the model gives a final answer and no longer asks to use tools.
### Usage ###
ResponseUsage(input_tokens=7110, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7220)


In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
len(chunks)

295

In [11]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

chunk_index.fit(chunks)

In [12]:
chunk_assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
)

In [13]:
chunk_answer, chunk_usage = chunk_assistant.rag("How does the agentic loop keep calling the model until it stops?")

print("### Output Text ###")
print(chunk_answer)
print("### Usage ###")
print(chunk_usage)

### Output Text ###
It keeps a `while True` loop and checks whether the model produced any `function_call` items in that turn.

- If there is at least one function call, the code runs the tool, appends the result, and loops again.
- If there are no function calls, it `break`s.

So the stopping condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps calling the model until a turn returns a final message with no tool calls.
### Usage ###
ResponseUsage(input_tokens=2293, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2400)


In [14]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [16]:
INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [19]:
def search(query: str) -> dict[str, str]:
    """
    Search the database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
    )

In [20]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [22]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [23]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
